In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find MedGemma-27b-text-it project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"

# Full path to the model snapshot
model_path = "/orcd/compute/mghassem/001/gobi1/huggingface/hub/models--google--medgemma-27b-text-it/snapshots/5b667cf2ddcf064085bc90952edb35a0edbfb79c"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

/home/yuexing/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 808/808 [01:20<00:00,  9.98it/s, Materializing param=model.norm.wei
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to understand, generate, and interact with human language.**

Think of them as incredibly sophisticated pattern-matching machines. They are trained on massive amounts of text data (like books, articles, websites) and learn the statistical relationships between words and concepts.

**Key characteristics:**

*   **"Large":** They have billions (or even trillions) of parameters, which are essentially the variables the model adjusts during training to learn patterns.
*   **"Language":** Their primary function is processing and generating human language.
*   **Capabilities:** They can perform tasks like:
    *   Answering questions
    *   Writing essays, code, or creative content
    *   Translating languages
    *   Summarizing text
    *   Holding conversations (like chatbots)

**In essence, LLMs are powerful tools that can mimic human-lik

In [ ]:
import pandas as pd 
import re
import torch

# Load data
df = pd.read_csv(paths.DATA / "Qwen72B_annotated_MedPAIR_relevancy.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None

# Create results dataframe
results = []

# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["Qwen72B_High_Relevance"]
        question = row["question_options_x"]
        
        # Improved prompt with clearer instructions
        query_full = (
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Based on the information provided, select the correct answer choice (A, B, C, D, etc.).\n\n"
            "IMPORTANT: Your response must end with 'Answer: X' where X is the letter of your chosen option.\n"
            "For example: Answer: A"
    )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
        )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
            qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(paths.PREDICTIONS / "[SR]_MedGemma27B_predictions_on_72B_progress.csv", index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "[SR]_MedGemma27B_predictions_72B.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21', 'Qwen72B_answer', 'Qwen72B_raw_response', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21', 'Qwen72B_High_Relevance', 'Match?', 'data_source_corr_trainee']
Processing 1300 rows...
Processing row 1/1300...
❌ Error on row 0: name 'qa_id' is not defined
Processing row 2/1300...
⚠️ Could not extract answer from response for row 2:
Response: **Thinki

✅ Processed Merge Q87: Answer = A
Saved progress to CSV after 90 items
Processing row 91/1300...
✅ Processed Merge Q87: Answer = B
Processing row 92/1300...
✅ Processed Merge Q87: Answer = C
Processing row 93/1300...
✅ Processed Merge Q87: Answer = B
Processing row 94/1300...
✅ Processed Merge Q87: Answer = C
Processing row 95/1300...
✅ Processed Merge Q87: Answer = J
Processing row 96/1300...
⚠️ Could not extract answer from response for row 96:
Response: **Analysis:**

1.  **Identify the key clinical information:** The patient is a 65-year-old male with...
✅ Processed Merge Q96: Answer = None
Processing row 97/1300...
✅ Processed Merge Q96: Answer = D
Processing row 98/1300...
✅ Processed Merge Q96: Answer = A
Processing row 99/1300...
✅ Processed Merge Q96: Answer = C
Processing row 100/1300...
✅ Processed Merge Q96: Answer = C
Saved progress to CSV after 100 items
Processing row 101/1300...
✅ Processed Merge Q96: Answer = C
Processing row 102/1300...
✅ Processed Merge Q96: Answer =

✅ Processed Merge Q185: Answer = I
Processing row 187/1300...


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "[SR]_MedGemma27B_predictions_72B.csv")
df = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_df3' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_df3' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_df3'] = df['data_source_df3']
else:
    print("Warning: 'data_source_df3' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_df3' in analysis_df.columns:
    category_stats = analysis_df.groupby('data_source_df3')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "[SR]_MedGemma27B_predictions_72B.csv")
df = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Count None/empty cells
def count_none_empty(series):
    """Count None, NaN, and empty string values"""
    return series.isna().sum() + (series == '').sum() + (series == 'None').sum()

# Overall None/empty counts
extracted_answer_none = count_none_empty(output_df['Extracted_Answer'])
answer_corr_none = count_none_empty(df['answer_corr'])

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print()
print(f"None/Empty in Extracted_Answer: {extracted_answer_none} ({extracted_answer_none/n*100:.2f}%)")
print(f"None/Empty in answer_corr: {answer_corr_none} ({answer_corr_none/n*100:.2f}%)")
print("=" * 60)
print()

# Analysis by category
if 'data_source_df3' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_df3' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_df3'] = df['data_source_df3']
else:
    print("Warning: 'data_source_df3' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_df3' in analysis_df.columns:
    # Add answer_corr to analysis_df for counting
    analysis_df['answer_corr'] = df['answer_corr']
    
    category_stats = analysis_df.groupby('data_source_df3').agg({
        'match': ['count', 'mean', 'std', lambda x: x.std() / np.sqrt(len(x))],
        'Extracted_Answer': lambda x: count_none_empty(x),
        'answer_corr': lambda x: count_none_empty(x)
    })
    
    # Flatten column names
    category_stats.columns = ['Count', 'Mean_Accuracy', 'Std_Dev', 'SE', 
                               'None_Empty_Extracted', 'None_Empty_Answer_Corr']
    category_stats = category_stats.reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    category_stats['None_Empty_Extracted_%'] = (category_stats['None_Empty_Extracted'] / category_stats['Count']) * 100
    category_stats['None_Empty_Answer_Corr_%'] = (category_stats['None_Empty_Answer_Corr'] / category_stats['Count']) * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 80)
    
    # Display main statistics
    display_cols = ['data_source_df3', 'Count', 'Mean_Accuracy_%', 'Std_Dev_%', 
                    'CI_95_Lower_%', 'CI_95_Upper_%']
    print(category_stats[display_cols].to_string(index=False))
    print()
    
    # Display None/Empty counts
    print("CATEGORY-WISE NONE/EMPTY COUNTS")
    print("-" * 80)
    none_cols = ['data_source_df3', 'Count', 'None_Empty_Extracted', 'None_Empty_Extracted_%',
                 'None_Empty_Answer_Corr', 'None_Empty_Answer_Corr_%']
    print(category_stats[none_cols].to_string(index=False))
    print("=" * 80)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 
               'Sample Size', 'None/Empty Extracted_Answer', 'None/Empty answer_corr'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n,
              f"{extracted_answer_none} ({extracted_answer_none/n*100:.2f}%)",
              f"{answer_corr_none} ({answer_corr_none/n*100:.2f}%)"]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)